**CREATE FACT TABLE**

In [0]:
%sql
select * from parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales/`;

In [0]:
df_silver = spark.sql("select * from parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales/`")
display(df_silver)

**Reading all the dims**

In [0]:
df_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")
df_branch = spark.sql("select * from cars_catalog.gold.dim_branch")
df_model = spark.sql("select * from cars_catalog.gold.dim_model")
df_date = spark.sql("select * from cars_catalog.gold.dim_date")

display(df_dealer.limit(5))
display(df_branch.limit(5))
display(df_model.limit(5))
display(df_date.limit(5))


**Bringing keys to the fact table**

In [0]:
df_fact = df_silver.join(df_branch, df_silver.Branch_ID == df_branch.Branch_ID, "left")\
    .join(df_dealer, df_silver.Dealer_ID == df_dealer.Dealer_ID, "left")\
        .join(df_model, df_silver.Model_ID == df_model.Model_ID, "left")\
            .join(df_date, df_silver.Date_ID == df_date.Date_ID, "left")\
                .select(df_silver.Revenue, df_silver.Units_Sold, df_silver.RevPerUnit, df_branch.dim_branch_key, df_dealer.dim_dealer_key, df_model.dim_model_key, df_date.dim_date_key)
display(df_fact)


In [0]:
from delta.tables import DeltaTable


In [0]:
%sql
DROP TABLE IF EXISTS cars_catalog.gold.factsales;


In [0]:
# Incremental Load
if spark.catalog.tableExists("cars_catalog.gold.factsales"):
    delta_tbl = DeltaTable.forName(spark, "cars_catalog.gold.factsales")

    delta_tbl.alias("trg").merge(
        df_fact.alias("src"),
        "trg.dim_date_key = src.dim_date_key AND "
        "trg.dim_branch_key = src.dim_branch_key AND "
        "trg.dim_dealer_key = src.dim_dealer_key AND "
        "trg.dim_model_key = src.dim_model_key"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

# Initial Load
else:
    df_fact.write.format("delta")\
        .mode("overwrite")\
        .option("path", "abfss://gold@carsamdatalake.dfs.core.windows.net/factsales")\
        .saveAsTable("cars_catalog.gold.factsales")


In [0]:
%sql
select * from cars_catalog.gold.factsales;